# Advanced Property Decorators — Tutorial-Style Problems with Solutions

This notebook is a **second, independent advanced problem set** on Python properties.

It intentionally follows a tutorial rhythm:

1. introduce one idea,
2. make a prediction,
3. build a small piece,
4. inspect what Python created,
5. extend the design,
6. test edge cases,
7. summarize the lesson.

The emphasis is not just on getting working code. The goal is to understand **what the `property` object is doing** and how to design property-based APIs carefully.

## What we will practice

We will repeatedly work with:

- `property(fget, fset, fdel, doc)`
- `@property`
- `@some_property.setter`
- `@some_property.getter`
- `@some_property.deleter`
- `fget`, `fset`, `fdel`, and `__doc__`
- validation before mutation
- backing attributes
- read-only, write-only, and read/write properties
- dependent invariants
- writable computed properties
- inheritance
- reusable property factories
- cache invalidation
- `__slots__`
- safe exposure of mutable data
- choosing between a property and a method

All code uses the Python standard library only.

## A recurring mental model

Suppose we write:

```python
class Example:
    @property
    def value(self):
        return self._value
```

Conceptually, Python evaluates the function first and then performs something equivalent to:

```python
value = property(value)
```

Later, if we write:

```python
@value.setter
def value(self, new_value):
    ...
```

the `setter(...)` method returns a **new property object** that contains the old getter plus the new setter, and that new property is rebound to the symbol `value`.

We will prove this several times instead of treating it as magic.

# Problem 1 — Build a Property in Slow Motion

We will start by avoiding decorator syntax completely.

Our goal is to build a `Score` class with a read/write `points` property.

Before writing the class, we will create the getter and setter functions separately so we can see exactly what a property object contains.

### Step 1 — Create ordinary functions

These are just functions. They are not properties yet.

Notice that both expect an instance as their first argument.

In [1]:
def get_points(self):
    return self._points


def set_points(self, value):
    self._points = value


print(get_points)
print(set_points)

<function get_points at 0x00000157923CD440>
<function set_points at 0x00000157923DD080>


### Step 2 — Create a getter-only property

`property(get_points)` returns a property object.

At this point:

- `fget` should be `get_points`
- `fset` should be `None`

In [2]:
getter_only = property(get_points)

print(type(getter_only))
print("fget:", getter_only.fget)
print("fset:", getter_only.fset)

assert getter_only.fget is get_points
assert getter_only.fset is None

<class 'property'>
fget: <function get_points at 0x00000157923CD440>
fset: None


### Step 3 — Add a setter

Now call the property's `.setter(...)` method.

Make a prediction before running the cell:

> Will `getter_only.setter(set_points)` modify the original property object, or return a new one?

In [3]:
read_write = getter_only.setter(set_points)

print("same object:", getter_only is read_write)
print("old id:", id(getter_only))
print("new id:", id(read_write))

assert getter_only is not read_write
assert read_write.fget is get_points
assert read_write.fset is set_points

same object: False
old id: 1475374512976
new id: 1475627353728


The important detail is that the original getter-only property still exists unchanged.

The new property carries forward the getter and adds the setter.

### Step 4 — Put the property on a class

Now we can assign the final property object to a class attribute.

In [4]:
class Score:
    points = read_write

    def __init__(self, points):
        self.points = points


s = Score(10)
print(s.points)

s.points = 25
print(s.points)

assert s.points == 25

10
25


### Step 5 — Rewrite the same class using decorators

This is much shorter, but the mechanism is the same.

In [5]:
class DecoratedScore:
    def __init__(self, points):
        self.points = points

    @property
    def points(self):
        return self._points

    @points.setter
    def points(self, value):
        self._points = value


ds = DecoratedScore(40)
ds.points = 50

assert ds.points == 50
print(ds.points)

50


### Takeaway

Decorator syntax is not a separate property system.

It is a convenient way to repeatedly replace a class symbol with the property object returned by `property(...)`, `.setter(...)`, `.getter(...)`, or `.deleter(...)`.

# Problem 2 — Why the Setter Name Matters

A subtle error happens when the setter function is given a different name.

We will deliberately create the bug, inspect the class namespace, and then fix it.

### Step 1 — Create the buggy class

The getter is named `reading`.

The decorated setter is named `set_reading`.

In [6]:
class Sensor:
    def __init__(self, reading):
        self._reading = reading

    @property
    def reading(self):
        return self._reading

    @reading.setter
    def set_reading(self, value):
        self._reading = value

### Step 2 — Inspect the class dictionary

We expect to find **two property objects**:

- `reading`
- `set_reading`

The first is still getter-only.
The second contains both the getter and setter.

In [7]:
for name, value in Sensor.__dict__.items():
    if isinstance(value, property):
        print(
            name,
            "getter =", getattr(value.fget, "__name__", None),
            "setter =", getattr(value.fset, "__name__", None),
        )

reading getter = reading setter = None
set_reading getter = reading setter = set_reading


### Step 3 — Test both attributes

In [8]:
sensor = Sensor(12)

print("reading:", sensor.reading)
print("set_reading getter:", sensor.set_reading)

try:
    sensor.reading = 99
except AttributeError as exc:
    print("sensor.reading assignment failed:", exc)

sensor.set_reading = 99

print("reading after setting through set_reading:", sensor.reading)
assert sensor.reading == 99

reading: 12
set_reading getter: 12
sensor.reading assignment failed: property 'reading' of 'Sensor' object has no setter
reading after setting through set_reading: 99


### Step 4 — Fix the class

The normal pattern is to reuse the same symbol.

In [9]:
class FixedSensor:
    def __init__(self, reading):
        self.reading = reading

    @property
    def reading(self):
        return self._reading

    @reading.setter
    def reading(self, value):
        self._reading = value


sensor = FixedSensor(1)
sensor.reading = 2

assert sensor.reading == 2
assert list(
    name for name, value in FixedSensor.__dict__.items()
    if isinstance(value, property)
) == ["reading"]

print(sensor.reading)

2


### Takeaway

The decorator expression uses the existing property object, but the **result is assigned to the function name that follows the decorator**.

That is why mismatched names can accidentally leave one getter-only property and create a second read/write property.

# Problem 3 — A Validated Network Port

Now we will use a setter for a realistic invariant.

A TCP/UDP port must be an integer in the range `1..65535`.

We want:

```python
config.port = 8080
```

to look like simple attribute assignment while still enforcing the rule.

### Step 1 — Decide the error types

We will use:

- `TypeError` when the value is not an integer
- `ValueError` when it is an integer but outside the valid range

We will also reject `bool`, because in Python `bool` is a subclass of `int`.

In [10]:
print(isinstance(True, int))

True


### Step 2 — Implement the property

In [11]:
class ServerConfig:
    def __init__(self, port):
        self.port = port

    @property
    def port(self):
        """TCP/UDP port number in the inclusive range 1..65535."""
        return self._port

    @port.setter
    def port(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("port must be an integer")

        if not 1 <= value <= 65535:
            raise ValueError("port must be in the range 1..65535")

        self._port = value

### Step 3 — Test normal and boundary values

In [12]:
for valid in (1, 80, 443, 65535):
    cfg = ServerConfig(valid)
    assert cfg.port == valid

print("valid boundaries passed")

valid boundaries passed


### Step 4 — Test invalid values

A failed assignment should not corrupt the previous valid value.

In [13]:
cfg = ServerConfig(8080)

for invalid in (0, 65536, -1):
    old = cfg.port

    try:
        cfg.port = invalid
    except ValueError as exc:
        print(invalid, "->", exc)

    assert cfg.port == old

for invalid in (3.14, "8080", True, None):
    old = cfg.port

    try:
        cfg.port = invalid
    except TypeError as exc:
        print(repr(invalid), "->", exc)

    assert cfg.port == old

0 -> port must be in the range 1..65535
65536 -> port must be in the range 1..65535
-1 -> port must be in the range 1..65535
3.14 -> port must be an integer
'8080' -> port must be an integer
True -> port must be an integer
None -> port must be an integer


### Takeaway

Validate first. Mutate second.

If a setter updates internal state before all checks are complete, a failed assignment can leave the object partially corrupted.

# Problem 4 — Normalize a Public Identifier

Properties can do more than reject data. They can also normalize it.

We will build `Repository.slug`.

Examples:

- `"  My Project  "` becomes `"my-project"`
- `"Python___Tips"` becomes `"python-tips"`

The public property always returns the normalized form.

### Step 1 — Write a normalization helper

Keeping transformation logic separate makes the setter easier to read and easier to test.

In [14]:
import re


def normalize_slug(value):
    if not isinstance(value, str):
        raise TypeError("slug must be a string")

    value = value.strip().lower()
    value = re.sub(r"[\s_]+", "-", value)
    value = re.sub(r"-+", "-", value)
    value = value.strip("-")

    if not value:
        raise ValueError("slug cannot be empty after normalization")

    if not re.fullmatch(r"[a-z0-9-]+", value):
        raise ValueError("slug contains unsupported characters")

    return value


for sample in ("  My Project  ", "Python___Tips", "a---b"):
    print(sample, "->", normalize_slug(sample))

  My Project   -> my-project
Python___Tips -> python-tips
a---b -> a-b


### Step 2 — Use the helper in a property

In [15]:
class Repository:
    def __init__(self, slug):
        self.slug = slug

    @property
    def slug(self):
        return self._slug

    @slug.setter
    def slug(self, value):
        self._slug = normalize_slug(value)


repo = Repository("  Advanced   Python ")
print(repo.slug)

assert repo.slug == "advanced-python"

advanced-python


### Step 3 — Change the value after construction

In [16]:
repo.slug = "Descriptors___And___Properties"

assert repo.slug == "descriptors-and-properties"
print(repo.slug)

descriptors-and-properties


### Takeaway

A property can present a clean public representation while hiding normalization rules and storage details.

# Problem 5 — A Two-Attribute Invariant

We now move from validating one value to validating **relationships between values**.

Create a `TimeWindow` such that:

```python
start <= end
```

must always remain true.

### Step 1 — Think about initialization

A common difficulty with dependent properties is that one property may not exist yet while the other is being assigned.

For this class, we will validate the pair first and then establish both backing attributes.

In [17]:
from datetime import datetime, timedelta

### Step 2 — Implement both managed endpoints

In [18]:
class TimeWindow:
    def __init__(self, start, end):
        if not isinstance(start, datetime) or not isinstance(end, datetime):
            raise TypeError("start and end must be datetime objects")

        if start > end:
            raise ValueError("start must not be later than end")

        self._start = start
        self._end = end

    @property
    def start(self):
        return self._start

    @start.setter
    def start(self, value):
        if not isinstance(value, datetime):
            raise TypeError("start must be a datetime")

        if value > self._end:
            raise ValueError("start must not be later than end")

        self._start = value

    @property
    def end(self):
        return self._end

    @end.setter
    def end(self, value):
        if not isinstance(value, datetime):
            raise TypeError("end must be a datetime")

        if value < self._start:
            raise ValueError("end must not be earlier than start")

        self._end = value

    @property
    def duration(self):
        return self._end - self._start

### Step 3 — Exercise the invariant

In [19]:
start = datetime(2026, 9, 11, 9, 0)
end = datetime(2026, 9, 11, 17, 0)

window = TimeWindow(start, end)

assert window.duration == timedelta(hours=8)
print(window.duration)

window.start = datetime(2026, 9, 11, 10, 0)
assert window.duration == timedelta(hours=7)

8:00:00


### Step 4 — Prove that invalid mutation is rejected without damaging the object

In [20]:
old_start = window.start

try:
    window.start = datetime(2026, 9, 11, 20, 0)
except ValueError as exc:
    print(exc)

assert window.start == old_start
assert window.start <= window.end

start must not be later than end


### Takeaway

When properties depend on one another, the class invariant matters more than any one setter.

A good setter must protect the **whole object**, not just the field it is changing.

# Problem 6 — A Writable Computed Property

Read-only computed properties are common.

A more advanced design is a **computed property with a setter**.

We will create a 2D vector with public attributes:

- `x`
- `y`
- `magnitude`

Reading `magnitude` computes `sqrt(x² + y²)`.

Assigning to `magnitude` rescales the vector while preserving its direction.

### Step 1 — Build the read-only version first

In [21]:
import math


class Vector2D:
    def __init__(self, x, y):
        self._x = float(x)
        self._y = float(y)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    @property
    def magnitude(self):
        return math.hypot(self._x, self._y)


v = Vector2D(3, 4)
assert v.magnitude == 5.0
print(v.magnitude)

5.0


### Step 2 — Add the setter

If the vector is non-zero, the scale factor is:

```text
new_magnitude / old_magnitude
```

Then multiply both coordinates by that factor.

The zero vector is a special case because its direction is undefined.

In [22]:
class RescalableVector2D:
    def __init__(self, x, y):
        self._x = float(x)
        self._y = float(y)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    @property
    def magnitude(self):
        return math.hypot(self._x, self._y)

    @magnitude.setter
    def magnitude(self, new_magnitude):
        if isinstance(new_magnitude, bool) or not isinstance(
            new_magnitude, (int, float)
        ):
            raise TypeError("magnitude must be numeric")

        if new_magnitude < 0:
            raise ValueError("magnitude cannot be negative")

        current = self.magnitude

        if current == 0:
            if new_magnitude == 0:
                return
            raise ValueError("cannot assign a positive magnitude to a zero vector")

        scale = float(new_magnitude) / current
        self._x *= scale
        self._y *= scale

### Step 3 — Test the rescaling

In [23]:
v = RescalableVector2D(3, 4)

v.magnitude = 10

print(v.x, v.y, v.magnitude)

assert abs(v.x - 6) < 1e-12
assert abs(v.y - 8) < 1e-12
assert abs(v.magnitude - 10) < 1e-12

6.0 8.0 10.0


### Step 4 — Test the special case

In [24]:
zero = RescalableVector2D(0, 0)

try:
    zero.magnitude = 5
except ValueError as exc:
    print(exc)

assert zero.magnitude == 0

cannot assign a positive magnitude to a zero vector


### Takeaway

A property's setter does not have to assign one backing attribute.

It can implement a higher-level semantic operation, as long as assignment remains understandable and unsurprising.

# Problem 7 — One Property Backed by Two Fields

Create `Employee.full_name`.

Internally we will store:

- `_first_name`
- `_last_name`

The getter combines them.

The setter parses a string and updates both fields.

### Step 1 — Define the parsing rule

For this exercise we require exactly two non-empty whitespace-separated parts.

In [25]:
def split_full_name(value):
    if not isinstance(value, str):
        raise TypeError("full_name must be a string")

    parts = value.split()

    if len(parts) != 2:
        raise ValueError("full_name must contain exactly first and last name")

    return parts[0], parts[1]


assert split_full_name("Ada Lovelace") == ("Ada", "Lovelace")

### Step 2 — Use the helper in a property

In [26]:
class Employee:
    def __init__(self, full_name):
        self.full_name = full_name

    @property
    def full_name(self):
        return f"{self._first_name} {self._last_name}"

    @full_name.setter
    def full_name(self, value):
        first, last = split_full_name(value)
        self._first_name = first
        self._last_name = last

    @property
    def first_name(self):
        return self._first_name

    @property
    def last_name(self):
        return self._last_name


employee = Employee("Ada Lovelace")

assert employee.first_name == "Ada"
assert employee.last_name == "Lovelace"
assert employee.full_name == "Ada Lovelace"

print(employee.full_name)

Ada Lovelace


### Step 3 — Reassign through the computed property

In [27]:
employee.full_name = "Grace Hopper"

assert employee.first_name == "Grace"
assert employee.last_name == "Hopper"

print(employee.first_name)
print(employee.last_name)

Grace
Hopper


### Takeaway

A property can act as a public view over multiple internal values.

The public API does not have to mirror the storage layout.

# Problem 8 — Normalize Paths and Derive Related Facts

We will create a `PathConfig.path` property using `pathlib.Path`.

The setter accepts either a string or `Path`.

The class also exposes read-only:

- `name`
- `suffix`
- `parent`

### Step 1 — Implement the property

In [28]:
from pathlib import Path


class PathConfig:
    def __init__(self, path):
        self.path = path

    @property
    def path(self):
        return self._path

    @path.setter
    def path(self, value):
        if not isinstance(value, (str, Path)):
            raise TypeError("path must be a string or pathlib.Path")

        candidate = Path(value).expanduser()

        if not candidate.name:
            raise ValueError("path must identify a file or directory name")

        self._path = candidate

    @property
    def name(self):
        return self._path.name

    @property
    def suffix(self):
        return self._path.suffix

    @property
    def parent(self):
        return self._path.parent

### Step 2 — Inspect the derived values

In [29]:
cfg = PathConfig("/tmp/archive.tar.gz")

print("path:", cfg.path)
print("name:", cfg.name)
print("suffix:", cfg.suffix)
print("parent:", cfg.parent)

assert cfg.name == "archive.tar.gz"
assert cfg.suffix == ".gz"

path: \tmp\archive.tar.gz
name: archive.tar.gz
suffix: .gz
parent: \tmp


### Step 3 — Reassign and confirm all derived properties change automatically

In [30]:
cfg.path = "/var/log/app.log"

assert cfg.name == "app.log"
assert cfg.suffix == ".log"

print(cfg.path, cfg.name, cfg.suffix)

\var\log\app.log app.log .log


### Takeaway

When a value is cheap to derive from canonical state, compute it on demand instead of duplicating it into additional attributes.

# Problem 9 — Use a Deleter as a Reset Operation

A deleter does not have to literally remove an attribute.

It can represent a meaningful reset operation.

We will build a `UserPreferences.language` property.

Deleting it will restore the default `"en"`.

### Step 1 — Implement getter, setter, and deleter

In [31]:
class UserPreferences:
    SUPPORTED = {"en", "bg", "de", "fr", "es"}

    def __init__(self, language="en"):
        self.language = language

    @property
    def language(self):
        """Preferred interface language."""
        return self._language

    @language.setter
    def language(self, value):
        if value not in self.SUPPORTED:
            raise ValueError(f"unsupported language: {value!r}")
        self._language = value

    @language.deleter
    def language(self):
        self._language = "en"

### Step 2 — Exercise the reset semantics

In [32]:
prefs = UserPreferences("bg")
assert prefs.language == "bg"

del prefs.language

assert prefs.language == "en"
print(prefs.language)

en


### Step 3 — Inspect the property object

In [33]:
language_property = UserPreferences.__dict__["language"]

print("getter:", language_property.fget)
print("setter:", language_property.fset)
print("deleter:", language_property.fdel)

assert language_property.fdel is not None

getter: <function UserPreferences.language at 0x00000157923DE3E0>
setter: <function UserPreferences.language at 0x00000157923DE8E0>
deleter: <function UserPreferences.language at 0x00000157923DEDE0>


### Takeaway

`del obj.attribute` can be given domain-specific meaning by a property deleter.

Use that carefully: the behavior should still be intuitive to someone reading the calling code.

# Problem 10 — A Write-Only PIN

The original property mechanism allows us to create a property without a getter.

We will use that to build a write-only `pin`.

The raw PIN will not be stored. We will store a hash and provide a `verify_pin(...)` method.

### Step 1 — Create an empty property with documentation

Because there is no getter, we provide `doc=...` directly to `property(...)`.

In [34]:
import hashlib
import hmac


class PinVault:
    pin = property(doc="Write-only PIN; the raw PIN is not stored.")

    def __init__(self, pin):
        self.pin = pin

    @pin.setter
    def pin(self, value):
        if not isinstance(value, str):
            raise TypeError("PIN must be a string")

        if not value.isdigit() or len(value) != 4:
            raise ValueError("PIN must contain exactly four digits")

        self._pin_digest = hashlib.sha256(value.encode("utf-8")).digest()

    def verify_pin(self, candidate):
        if not isinstance(candidate, str):
            return False

        digest = hashlib.sha256(candidate.encode("utf-8")).digest()
        return hmac.compare_digest(self._pin_digest, digest)

### Step 2 — Verify normal behavior

In [35]:
vault = PinVault("1234")

assert vault.verify_pin("1234")
assert not vault.verify_pin("0000")

print(vault.verify_pin("1234"))

True


### Step 3 — Attempt to read the property

Because no getter exists, reading is an error.

In [36]:
try:
    print(vault.pin)
except AttributeError as exc:
    print("cannot read pin:", exc)

print("property documentation:", PinVault.pin.__doc__)

cannot read pin: property 'pin' of 'PinVault' object has no getter
property documentation: Write-only PIN; the raw PIN is not stored.


### Security note

A write-only property is not, by itself, a security boundary.

The stronger part of this design is that the raw PIN is not retained in the instance.

# Problem 11 — Where Does Property Documentation Come From?

We will compare three cases:

1. docstring on the getter,
2. docstring only on the setter,
3. explicit `doc=` supplied to `property(...)`.

### Case 1 — Getter docstring

In [37]:
class GetterDocumented:
    @property
    def value(self):
        """Documentation from the getter."""
        return 1


print(GetterDocumented.value.__doc__)
assert GetterDocumented.value.__doc__ == "Documentation from the getter."

Documentation from the getter.


### Case 2 — Setter-only docstring

In [38]:
class SetterDocumented:
    def __init__(self):
        self._value = 1

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        """Documentation placed on the setter."""
        self._value = new_value


print("property doc:", SetterDocumented.value.__doc__)
print("setter doc:", SetterDocumented.value.fset.__doc__)

assert SetterDocumented.value.__doc__ is None
assert SetterDocumented.value.fset.__doc__ is not None

property doc: None
setter doc: Documentation placed on the setter.


### Case 3 — Explicit property documentation

In [39]:
class ExplicitlyDocumented:
    value = property(
        fget=lambda self: 1,
        doc="Documentation explicitly supplied to property(...).",
    )


print(ExplicitlyDocumented.value.__doc__)

Documentation explicitly supplied to property(...).


### Takeaway

For normal decorator-style read properties, put public documentation in the getter.

For a property without a getter, use the `doc=` argument.

# Problem 12 — Override Only One Accessor in a Subclass

Properties are objects, so subclasses can reuse one accessor while replacing another.

We will create:

- `Document.title`: accepts any non-empty title
- `PublishedDocument.title`: uses the same getter but rejects titles longer than 60 characters

### Step 1 — Base class

In [40]:
class Document:
    def __init__(self, title):
        self.title = title

    @property
    def title(self):
        return self._title

    @title.setter
    def title(self, value):
        if not isinstance(value, str):
            raise TypeError("title must be a string")

        value = value.strip()

        if not value:
            raise ValueError("title cannot be empty")

        self._title = value

### Step 2 — Subclass only the setter

The expression before `.setter` is the base-class property object.

In [41]:
class PublishedDocument(Document):
    @Document.title.setter
    def title(self, value):
        if not isinstance(value, str):
            raise TypeError("title must be a string")

        value = value.strip()

        if not value:
            raise ValueError("title cannot be empty")

        if len(value) > 60:
            raise ValueError("published title cannot exceed 60 characters")

        self._title = value

### Step 3 — Compare the two property objects

In [42]:
print("same getter:",
      Document.title.fget is PublishedDocument.title.fget)

print("same setter:",
      Document.title.fset is PublishedDocument.title.fset)

assert Document.title.fget is PublishedDocument.title.fget
assert Document.title.fset is not PublishedDocument.title.fset

same getter: True
same setter: False


### Step 4 — Test the specialization

In [43]:
draft = Document("x" * 100)
assert len(draft.title) == 100

try:
    PublishedDocument("x" * 100)
except ValueError as exc:
    print(exc)

published = PublishedDocument("A concise title")
print(published.title)

published title cannot exceed 60 characters
A concise title


### Takeaway

`@BaseClass.some_property.setter` creates a new property using the base getter and a replacement setter.

This can be cleaner than rewriting the getter just to change validation.

# Problem 13 — Write a Property Factory

If several attributes follow the same property rules, a factory can generate property objects.

We will create a reusable `bounded_float(...)` function.

### Step 1 — Define the factory

The factory receives:

- the public attribute name,
- minimum value,
- maximum value.

It returns a property that stores into `_<name>`.

In [44]:
def bounded_float(name, minimum, maximum):
    private_name = "_" + name

    def getter(self):
        return getattr(self, private_name)

    def setter(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{name} must be numeric")

        value = float(value)

        if not minimum <= value <= maximum:
            raise ValueError(
                f"{name} must be between {minimum} and {maximum}"
            )

        setattr(self, private_name, value)

    return property(
        getter,
        setter,
        doc=f"{name} constrained to [{minimum}, {maximum}]",
    )

### Step 2 — Use the factory in a geographic coordinate class

In [45]:
class GeoPoint:
    latitude = bounded_float("latitude", -90, 90)
    longitude = bounded_float("longitude", -180, 180)

    def __init__(self, latitude, longitude):
        self.latitude = latitude
        self.longitude = longitude


point = GeoPoint(42.7, 23.3)

print(point.latitude, point.longitude)
print(GeoPoint.latitude.__doc__)

42.7 23.3
latitude constrained to [-90, 90]


### Step 3 — Test invalid data

In [46]:
try:
    point.latitude = 100
except ValueError as exc:
    print(exc)

try:
    point.longitude = "east"
except TypeError as exc:
    print(exc)

latitude must be between -90 and 90
longitude must be numeric


### Takeaway

Factories can reduce duplicated accessor logic.

They are most useful when the repeated behavior is stable and truly identical across several attributes.

# Problem 14 — Cache a Derived Property and Invalidate It Correctly

Caching introduces a new responsibility:

> If source data changes, every cached value derived from it must become invalid.

We will build a `Polyline` whose `length` is cached.

### Step 1 — Decide the canonical state

The canonical state is a tuple of `(x, y)` points.

The cached value is `_cached_length`.

In [47]:
class Polyline:
    def __init__(self, points):
        self.points = points

    @property
    def points(self):
        return self._points

    @points.setter
    def points(self, value):
        converted = tuple(
            (float(x), float(y))
            for x, y in value
        )

        if len(converted) < 2:
            raise ValueError("a polyline needs at least two points")

        self._points = converted

        if hasattr(self, "_cached_length"):
            del self._cached_length

    @property
    def length(self):
        if not hasattr(self, "_cached_length"):
            total = 0.0

            for (x1, y1), (x2, y2) in zip(
                self._points,
                self._points[1:],
            ):
                total += math.hypot(x2 - x1, y2 - y1)

            self._cached_length = total

        return self._cached_length

### Step 2 — Read the property twice

The first access computes it.

The second access reuses the cached value.

In [48]:
line = Polyline([(0, 0), (3, 4), (6, 8)])

first = line.length
second = line.length

print(first, second)
assert first == 10.0
assert second == 10.0
assert hasattr(line, "_cached_length")

10.0 10.0


### Step 3 — Change the source data

The setter must delete the stale cache.

In [49]:
line.points = [(0, 0), (0, 2)]

assert not hasattr(line, "_cached_length")

print("recomputed:", line.length)
assert line.length == 2.0

recomputed: 2.0


### Takeaway

Caching is not just a getter optimization.

It couples the getter to every mutation path that can affect the computed value.

# Problem 15 — Properties with `__slots__`

A property does not require instances to have a `__dict__`.

We will create a slotted `ColorChannel` class with managed properties.

### Step 1 — Define the slots

The slot names are the backing attributes, not the public property names.

In [50]:
class ColorChannel:
    __slots__ = ("_red", "_green", "_blue")

    def __init__(self, red, green, blue):
        self.red = red
        self.green = green
        self.blue = blue

    @staticmethod
    def _channel(value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("channel must be an integer")
        if not 0 <= value <= 255:
            raise ValueError("channel must be in the range 0..255")
        return value

    @property
    def red(self):
        return self._red

    @red.setter
    def red(self, value):
        self._red = self._channel(value)

    @property
    def green(self):
        return self._green

    @green.setter
    def green(self, value):
        self._green = self._channel(value)

    @property
    def blue(self):
        return self._blue

    @blue.setter
    def blue(self, value):
        self._blue = self._channel(value)

### Step 2 — Verify that normal property syntax still works

In [51]:
c = ColorChannel(10, 20, 30)

c.red = 100

assert c.red == 100
assert c.green == 20
assert c.blue == 30

print(c.red, c.green, c.blue)

100 20 30


### Step 3 — Verify there is no instance dictionary

In [52]:
print("has __dict__:", hasattr(c, "__dict__"))
assert not hasattr(c, "__dict__")

has __dict__: False


### Takeaway

Properties operate through descriptor behavior on the class.

They can manage storage in normal instance dictionaries, slots, or other mechanisms.

# Problem 16 — Read-Only Property vs Mutable Object

A property with no setter is only read-only at the **attribute assignment** level.

If it returns a mutable object, callers may still mutate that object.

### Step 1 — Demonstrate the unsafe version

In [53]:
class TeamUnsafe:
    def __init__(self, members):
        self._members = set(members)

    @property
    def members(self):
        return self._members


team = TeamUnsafe({"Ada", "Grace"})

team.members.add("Guido")

print(team.members)
assert "Guido" in team.members

{'Guido', 'Ada', 'Grace'}


The property is getter-only, but we still changed internal state.

So "read-only property" does not automatically mean "immutable state."

### Step 2 — Return an immutable view

In [54]:
class TeamSafe:
    def __init__(self, members):
        self._members = set(members)

    @property
    def members(self):
        return frozenset(self._members)

    def add_member(self, name):
        self._members.add(name)


team = TeamSafe({"Ada", "Grace"})

snapshot = team.members

print(snapshot)
assert isinstance(snapshot, frozenset)

frozenset({'Ada', 'Grace'})


### Step 3 — Mutation now goes through an explicit method

In [55]:
team.add_member("Guido")

assert "Guido" in team.members
print(team.members)

frozenset({'Guido', 'Ada', 'Grace'})


### Takeaway

A getter-only property controls rebinding:

```python
obj.members = something
```

It does not control mutation of an object returned by the getter.

Return immutable values, copies, or read-only views when that distinction matters.

# Problem 17 — Property or Method?

Not every operation belongs in a property.

We will design a `Download` object.

Good property candidates:

- `bytes_received`
- `total_bytes`
- `progress`

Good method candidates:

- `receive(chunk_size)`
- `reset()`

Why?

`progress` is an attribute-like fact.

`receive(...)` is an action with a meaningful side effect.

### Step 1 — Implement managed state

In [56]:
class Download:
    def __init__(self, total_bytes):
        self.total_bytes = total_bytes
        self._bytes_received = 0

    @property
    def total_bytes(self):
        return self._total_bytes

    @total_bytes.setter
    def total_bytes(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("total_bytes must be an integer")

        if value <= 0:
            raise ValueError("total_bytes must be positive")

        if hasattr(self, "_bytes_received") and self._bytes_received > value:
            raise ValueError(
                "total_bytes cannot be below bytes already received"
            )

        self._total_bytes = value

    @property
    def bytes_received(self):
        return self._bytes_received

    @property
    def progress(self):
        return self._bytes_received / self._total_bytes

    def receive(self, chunk_size):
        if isinstance(chunk_size, bool) or not isinstance(chunk_size, int):
            raise TypeError("chunk_size must be an integer")

        if chunk_size <= 0:
            raise ValueError("chunk_size must be positive")

        new_total = self._bytes_received + chunk_size

        if new_total > self._total_bytes:
            raise ValueError("received bytes would exceed total size")

        self._bytes_received = new_total

    def reset(self):
        self._bytes_received = 0

### Step 2 — Use the API

In [57]:
download = Download(1000)

download.receive(250)
print(download.progress)

download.receive(250)
print(download.progress)

assert download.bytes_received == 500
assert download.progress == 0.5

0.25
0.5


### Step 3 — Reset is an operation, so make it explicit

In [58]:
download.reset()

assert download.bytes_received == 0
assert download.progress == 0.0

print(download.progress)

0.0


### Takeaway

Properties work best for values that feel like attributes.

Methods are clearer for commands, workflows, or operations that carry domain meaning.

# Problem 18 — Write-Once Then Read-Only

Sometimes an attribute should be set during construction and never changed again.

We will build `Message.message_id` as a write-once property.

### Step 1 — Detect whether the backing value already exists

In [59]:
class Message:
    def __init__(self, message_id, body):
        self.message_id = message_id
        self.body = body

    @property
    def message_id(self):
        return self._message_id

    @message_id.setter
    def message_id(self, value):
        if hasattr(self, "_message_id"):
            raise AttributeError("message_id is write-once")

        if not isinstance(value, str) or not value.strip():
            raise ValueError("message_id must be a non-empty string")

        self._message_id = value.strip()

    @property
    def body(self):
        return self._body

    @body.setter
    def body(self, value):
        if not isinstance(value, str):
            raise TypeError("body must be a string")

        self._body = value

### Step 2 — Verify that the first assignment works

In [60]:
message = Message("msg-001", "Hello")

assert message.message_id == "msg-001"
print(message.message_id)

msg-001


### Step 3 — Verify that later assignment fails

In [61]:
try:
    message.message_id = "msg-002"
except AttributeError as exc:
    print(exc)

assert message.message_id == "msg-001"

message_id is write-once


### Takeaway

A setter can distinguish initialization from later mutation.

Use this pattern selectively. If an object is broadly immutable, a frozen dataclass or another immutable design may be clearer.

# Problem 19 — Transactional Setter Logic

A setter may need several validation steps.

The safest pattern is:

1. parse into local variables,
2. validate everything,
3. only then change object state.

We will implement a `Version.version` property for semantic versions such as `"2.14.3"`.

### Step 1 — Parse without mutating the object

In [62]:
def parse_version(value):
    if not isinstance(value, str):
        raise TypeError("version must be a string")

    parts = value.split(".")

    if len(parts) != 3:
        raise ValueError("version must have major.minor.patch")

    if not all(part.isdigit() for part in parts):
        raise ValueError("version components must be non-negative integers")

    major, minor, patch = map(int, parts)

    return major, minor, patch


assert parse_version("2.14.3") == (2, 14, 3)

### Step 2 — Commit only after parsing succeeds

In [63]:
class Version:
    def __init__(self, version):
        self.version = version

    @property
    def version(self):
        return f"{self._major}.{self._minor}.{self._patch}"

    @version.setter
    def version(self, value):
        major, minor, patch = parse_version(value)

        self._major = major
        self._minor = minor
        self._patch = patch

    @property
    def tuple(self):
        return self._major, self._minor, self._patch

### Step 3 — Prove failed assignment preserves old state

In [64]:
v = Version("1.2.3")

old = v.tuple

try:
    v.version = "2.bad.0"
except ValueError as exc:
    print(exc)

assert v.tuple == old
assert v.version == "1.2.3"

version components must be non-negative integers


### Takeaway

Treat complicated setters like small transactions.

Do all risky work first. Commit state only after the new value has been fully validated.

# Problem 20 — Capstone: Subscription Configuration

We will combine many of the ideas from the notebook.

Create a `Subscription` with:

- `plan`: `"free"`, `"pro"`, or `"enterprise"`
- `seats`: positive integer
- `monthly_price`: derived, read-only
- `annual_price`: derived, read-only
- `billing_email`: normalized and validated
- `external_id`: write-once
- `coupon_percent`: validated `0..100`
- deleting `coupon_percent` resets it to `0`
- `summary`: read-only formatted text

There is also a cross-field rule:

- the `"free"` plan must always have exactly `1` seat

Changing either `plan` or `seats` must preserve that invariant.

## Capstone Step 1 — Decide the pricing model

For this exercise:

- `free`: `$0` per seat/month
- `pro`: `$25` per seat/month
- `enterprise`: `$80` per seat/month

Annual billing is twelve months after applying the coupon.

In [65]:
PLAN_PRICE = {
    "free": 0.0,
    "pro": 25.0,
    "enterprise": 80.0,
}

## Capstone Step 2 — Implement the class skeleton

We will establish the two interdependent values (`plan` and `seats`) carefully during construction.

In [66]:
class Subscription:
    def __init__(
        self,
        external_id,
        plan,
        seats,
        billing_email,
        coupon_percent=0,
    ):
        if plan not in PLAN_PRICE:
            raise ValueError("unknown plan")

        if isinstance(seats, bool) or not isinstance(seats, int):
            raise TypeError("seats must be an integer")

        if seats <= 0:
            raise ValueError("seats must be positive")

        if plan == "free" and seats != 1:
            raise ValueError("free plan must have exactly one seat")

        self._plan = plan
        self._seats = seats

        self.external_id = external_id
        self.billing_email = billing_email
        self.coupon_percent = coupon_percent

## Capstone Step 3 — Add the write-once external ID

In [67]:
def _get_external_id(self):
    return self._external_id


def _set_external_id(self, value):
    if hasattr(self, "_external_id"):
        raise AttributeError("external_id is write-once")

    if not isinstance(value, str) or not value.strip():
        raise ValueError("external_id must be a non-empty string")

    self._external_id = value.strip()


Subscription.external_id = property(
    _get_external_id,
    _set_external_id,
    doc="Write-once external subscription identifier.",
)

We assigned this property after the class body on purpose.

That reinforces the fact that a property is just a class-level object. Decorator syntax is convenient, but not required.

## Capstone Step 4 — Add `plan` with cross-field validation

In [68]:
def get_plan(self):
    return self._plan


def set_plan(self, value):
    if value not in PLAN_PRICE:
        raise ValueError("unknown plan")

    if value == "free" and self._seats != 1:
        raise ValueError("cannot switch to free unless seats == 1")

    self._plan = value


Subscription.plan = property(
    get_plan,
    set_plan,
    doc="Current subscription plan.",
)

## Capstone Step 5 — Add `seats` with the matching invariant

In [69]:
def get_seats(self):
    return self._seats


def set_seats(self, value):
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError("seats must be an integer")

    if value <= 0:
        raise ValueError("seats must be positive")

    if self._plan == "free" and value != 1:
        raise ValueError("free plan must have exactly one seat")

    self._seats = value


Subscription.seats = property(
    get_seats,
    set_seats,
    doc="Number of licensed seats.",
)

## Capstone Step 6 — Add normalized billing email

In [70]:
def get_billing_email(self):
    return self._billing_email


def set_billing_email(self, value):
    if not isinstance(value, str):
        raise TypeError("billing_email must be a string")

    candidate = value.strip().lower()

    if candidate.count("@") != 1:
        raise ValueError("invalid billing email")

    local, domain = candidate.split("@")

    if not local or "." not in domain:
        raise ValueError("invalid billing email")

    self._billing_email = candidate


Subscription.billing_email = property(
    get_billing_email,
    set_billing_email,
    doc="Normalized billing contact email.",
)

## Capstone Step 7 — Add coupon getter, setter, and deleter

In [71]:
def get_coupon_percent(self):
    return self._coupon_percent


def set_coupon_percent(self, value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError("coupon_percent must be numeric")

    value = float(value)

    if not 0 <= value <= 100:
        raise ValueError("coupon_percent must be in [0, 100]")

    self._coupon_percent = value


def delete_coupon_percent(self):
    self._coupon_percent = 0.0


Subscription.coupon_percent = property(
    get_coupon_percent,
    set_coupon_percent,
    delete_coupon_percent,
    "Discount percentage applied to subscription pricing.",
)

## Capstone Step 8 — Add derived read-only properties

These values are cheap to compute, so we do not cache them.

In [72]:
def get_monthly_price(self):
    base = PLAN_PRICE[self._plan] * self._seats
    multiplier = 1 - self._coupon_percent / 100
    return base * multiplier


def get_annual_price(self):
    return self.monthly_price * 12


def get_summary(self):
    return (
        f"{self._external_id}: "
        f"{self._plan}, "
        f"{self._seats} seat(s), "
        f"${self.monthly_price:.2f}/month"
    )


Subscription.monthly_price = property(
    get_monthly_price,
    doc="Monthly price after coupon discount.",
)

Subscription.annual_price = property(
    get_annual_price,
    doc="Annual price after coupon discount.",
)

Subscription.summary = property(
    get_summary,
    doc="Human-readable subscription summary.",
)

## Capstone Step 9 — Test the normal path

In [73]:
sub = Subscription(
    external_id=" sub-100 ",
    plan="pro",
    seats=3,
    billing_email=" BILLING@Example.COM ",
    coupon_percent=20,
)

print(sub.summary)
print("monthly:", sub.monthly_price)
print("annual:", sub.annual_price)

assert sub.external_id == "sub-100"
assert sub.billing_email == "billing@example.com"
assert sub.monthly_price == 60.0
assert sub.annual_price == 720.0

sub-100: pro, 3 seat(s), $60.00/month
monthly: 60.0
annual: 720.0


## Capstone Step 10 — Test independent mutations

In [74]:
sub.seats = 4
assert sub.monthly_price == 80.0

sub.coupon_percent = 50
assert sub.monthly_price == 50.0

print(sub.summary)

sub-100: pro, 4 seat(s), $50.00/month


## Capstone Step 11 — Test the cross-field invariant

A pro subscription with multiple seats cannot directly become free.

In [75]:
old_plan = sub.plan

try:
    sub.plan = "free"
except ValueError as exc:
    print(exc)

assert sub.plan == old_plan

cannot switch to free unless seats == 1


To move to the free plan, first reduce seats to one and then change the plan.

In [76]:
sub.seats = 1
sub.plan = "free"

assert sub.plan == "free"
assert sub.seats == 1
assert sub.monthly_price == 0.0

print(sub.summary)

sub-100: free, 1 seat(s), $0.00/month


Now that the plan is free, increasing seats must fail.

In [77]:
try:
    sub.seats = 2
except ValueError as exc:
    print(exc)

assert sub.seats == 1

free plan must have exactly one seat


## Capstone Step 12 — Test deleter and write-once behavior

In [78]:
del sub.coupon_percent

assert sub.coupon_percent == 0.0

try:
    sub.external_id = "sub-999"
except AttributeError as exc:
    print(exc)

assert sub.external_id == "sub-100"

external_id is write-once


## Capstone Step 13 — Introspect the generated properties

Because some of these properties were attached after the class body, inspecting the class makes the mechanism especially visible.

In [79]:
for name in (
    "external_id",
    "plan",
    "seats",
    "billing_email",
    "coupon_percent",
    "monthly_price",
    "annual_price",
    "summary",
):
    prop = Subscription.__dict__[name]

    print(
        f"{name:16}",
        "get =", prop.fget is not None,
        "set =", prop.fset is not None,
        "del =", prop.fdel is not None,
    )

external_id      get = True set = True del = False
plan             get = True set = True del = False
seats            get = True set = True del = False
billing_email    get = True set = True del = False
coupon_percent   get = True set = True del = True
monthly_price    get = True set = False del = False
annual_price     get = True set = False del = False
summary          get = True set = False del = False


### Capstone takeaway

The capstone used properties in several distinct ways:

- validation
- normalization
- write-once semantics
- cross-field invariant protection
- derived values
- reset-through-deletion
- explicit property construction
- introspection

The same `property` mechanism supports all of these patterns.

# Extra Guided Mini-Problems

The next exercises are shorter, but still include solutions.

Try to predict each result before running the solution cell.

## Mini-Problem A — Can a setter change another attribute?

Create `Square.side` such that changing the side also increments a `revision` counter.

This is valid because a setter is ordinary Python code.

In [80]:
class Square:
    def __init__(self, side):
        self._revision = 0
        self.side = side

    @property
    def side(self):
        return self._side

    @side.setter
    def side(self, value):
        if value <= 0:
            raise ValueError("side must be positive")

        self._side = float(value)
        self._revision += 1

    @property
    def revision(self):
        return self._revision


sq = Square(2)
assert sq.revision == 1

sq.side = 3
assert sq.revision == 2

print(sq.side, sq.revision)

3.0 2


## Mini-Problem B — Can a property getter raise intentionally?

Yes.

Create `Session.user` so it raises `RuntimeError` when the session is anonymous.

In [81]:
class Session:
    def __init__(self):
        self._user = None

    @property
    def user(self):
        if self._user is None:
            raise RuntimeError("anonymous session has no user")
        return self._user

    def login(self, user):
        self._user = user

    def logout(self):
        self._user = None


session = Session()

try:
    session.user
except RuntimeError as exc:
    print(exc)

session.login("Ada")
assert session.user == "Ada"

anonymous session has no user


## Mini-Problem C — Can two properties share the same backing attribute?

Yes.

Create `Angle.degrees` and `Angle.radians`, both backed by `_radians`.

In [82]:
class Angle:
    def __init__(self, degrees):
        self.degrees = degrees

    @property
    def degrees(self):
        return math.degrees(self._radians)

    @degrees.setter
    def degrees(self, value):
        self._radians = math.radians(float(value))

    @property
    def radians(self):
        return self._radians

    @radians.setter
    def radians(self, value):
        self._radians = float(value)


angle = Angle(180)

assert abs(angle.radians - math.pi) < 1e-12

angle.radians = math.pi / 2

assert abs(angle.degrees - 90) < 1e-12

print(angle.degrees, angle.radians)

90.0 1.5707963267948966


## Mini-Problem D — Does a property have to use an underscore attribute?

No.

The underscore convention is common and useful, but storage can be elsewhere.

Here we store values in an internal dictionary.

In [83]:
class FlexibleRecord:
    def __init__(self, code):
        self._data = {}
        self.code = code

    @property
    def code(self):
        return self._data["code"]

    @code.setter
    def code(self, value):
        if not isinstance(value, str) or not value:
            raise ValueError("code must be a non-empty string")

        self._data["code"] = value


record = FlexibleRecord("A1")
record.code = "B2"

assert record.code == "B2"
print(record._data)

{'code': 'B2'}


# Final Review

A useful way to reason about properties is to separate three questions.

## 1. What is the public interface?

Examples:

```python
obj.port
obj.port = 8080

obj.magnitude
obj.magnitude = 10

del obj.language
```

## 2. What invariant or behavior does the class promise?

Examples:

- a port is always in `1..65535`
- a time window always satisfies `start <= end`
- a free subscription always has exactly one seat
- a write-once ID never changes
- a cached result is invalidated when its source changes

## 3. Where and how is state actually stored?

Examples:

- `_port`
- `_start` / `_end`
- several internal fields
- a dictionary
- slotted backing attributes
- a canonical unit such as radians

A property connects the public interface to the internal representation.

# Best-Practice Checklist

Before adding a property, ask:

- Is this value naturally attribute-like?
- Do I need validation, normalization, conversion, or controlled mutation?
- Is a method clearer because the operation represents an action?
- Am I validating everything before mutating state?
- Does a failed setter leave the old valid state intact?
- Do multiple properties need to preserve a shared invariant?
- Am I accidentally returning mutable internal state?
- If I cache, where is the cache invalidated?
- If I use inheritance, can I reuse the base getter or setter?
- Is the property documentation attached to the getter or supplied explicitly?
- Have I remembered that `.setter`, `.getter`, and `.deleter` return new property objects?

# Suggested Follow-Up Practice

Rebuild these without looking at the solutions:

1. `ServerConfig.port`
2. `TimeWindow`
3. writable `Vector2D.magnitude`
4. `PinVault`
5. `PublishedDocument`
6. `GeoPoint` using a property factory
7. cached `Polyline.length`
8. the `Subscription` capstone

Then modify at least three of them so that a failed assignment is tested for **state preservation**, not only for the correct exception.